# Workflow H3 end-to-end e benchmark

Questo notebook descrive le procedure piu' recenti per scaricare anni selezionati, preprocessarli, finalizzare `census_data.gpkg`, stimare layer H3 e confrontare impostazioni di parallelizzazione.

Le celle operative complete sono protette da variabili booleane perche' possono scaricare dati o richiedere GeoPackage locali di grandi dimensioni.


In [ ]:
from pathlib import Path

import geopandas as gpd

from istat_census_data import run_h3_census_workflow


## Workflow end-to-end da Python


In [ ]:
RUN_END_TO_END_WORKFLOW = False

data_folder = Path("census_tutorial")
aoi_path = Path("aoi.gpkg")

if RUN_END_TO_END_WORKFLOW:
    aoi = gpd.read_file(aoi_path, layer="aoi")
    workflow_result = run_h3_census_workflow(
        years=[2021],
        data_folder=data_folder,
        aoi=aoi,
        indexes=["P1", "P2"],
        region_list=[15],
        workers=4,
        chunk_size=5000,
        parallel_backend="thread",
    )
    print(workflow_result.output_path)
    print(workflow_result.layers)
else:
    print("Workflow saltato. Imposta RUN_END_TO_END_WORKFLOW = True dopo aver preparato un AOI.")


## Workflow end-to-end dallo script

`scripts/run_h3_census_analysis.py` e' uno script di esempio modificabile. Imposta i valori di modulo come `list_year`, `main_path`, `aoi_path`, `list_region`, `selected_indexes`, `workers`, `chunk_size` e `parallel_backend`, poi esegui lo script dalla cartella del progetto.

Usa `selected_indexes = []` quando vuoi tutti gli indicatori additivi del catalogo. Imposta `run_download`, `run_preprocess` o `run_finalize` a `False` quando i file necessari esistono gia' localmente.


## Benchmark delle impostazioni worker/chunk

Il benchmark deve usare dati finalizzati locali. Modifica `scripts/benchmark_h3_estimation.py`, parti da pochi `worker_counts`, confronta percorso seriale e parallelo e salva l'output JSON quando vuoi confrontare piu' run. Mantieni `selected_indexes = []` per stimare tutti gli indicatori additivi del catalogo.


## Backend Dask opzionale

Installa l'extra opzionale prima di usare Dask:

```bash
pip install "istat-census-data[dask]"
```

Poi usa `parallel_backend="dask"` in Python oppure `parallel_backend = "dask"` negli script.
